# 📦 Step 1 — 下载 Qwen3.8-27B 去审查版 GGUF 并打成 Kaggle Dataset

这个 notebook **只跑一次**。它把 Qwen3.8-27B 去审查版（abliterated）的 GGUF 直接从 HuggingFace 拉到 Kaggle 机器上，然后打包成一个**永久的私有 Kaggle Dataset**。

数据集建好之后，`2_run_server.ipynb` 每次开跑只要挂载它就行 —— 再也不用重下。

---

### 下载什么

| 文件 | 大小 | 用途 |
|---|---|---|
| `Huihui-Qwen3.8-27B-abliterated-UD-Q5_K_XL.gguf` | 19.25 GB | 主模型（unsloth 动态量化 Q5） |
| `mmproj-model-bf16.gguf` | 0.87 GB | 视觉投影层 —— 图片输入必需 |

合计约 **20.1 GB**，Kaggle 的 100 GB 配额毫无压力。

### 为什么是这个 repo

`huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF` 是标准的 abliteration（权重正交化）去审查版：
- 直接对 Qwen 官方 3.8-27B 做偏置消融，**不动底座能力**，不是重训练的小作坊微调
- 保留了原生 262,144 上下文、视觉理解、thinking 开关、MTP 头
- 量化档最全（IQ2 ~ Q8 全系）

> 同仓库还有 `-MTP` 后缀的变体（自带 MTP 投机头，能提速 1.5~1.8x），但只有 IQ2/IQ3 档有，Q5 没有。想省显存换速度的话把下面的 `QUANT` 改成 `UD-IQ3_S-MTP`（11.13 GB）。

---

### 跑之前的检查

1. Kaggle 右侧设置 → **Internet = ON**（必须，要下 HF）
2. 这个 notebook **不需要 GPU**，CPU 就跑得动
3. 账号剩余配额 ≥ 21 GB（Kaggle 给 100 GB）

## ⚙️ Cell 1 — Install dependencies

We install two packages:
- `huggingface_hub` — the official HuggingFace download library
- `hf_transfer` — a fast Rust-based transfer backend that makes large downloads **3-5x faster**

In [ ]:
%%capture
!pip install -q huggingface_hub hf_transfer

## ⚙️ Cell 2 — Enable fast transfer & define paths

Setting `HF_XET_HIGH_PERFORMANCE=1` unlocks the high-performance Xet transfer backend.
(`HF_HUB_ENABLE_HF_TRANSFER` is deprecated in current `huggingface_hub` — hf_transfer is no longer used —
so it only produces a `FutureWarning` and does nothing.)

We download into `/tmp/model/` instead of `/kaggle/working/`.

> **Why `/tmp` and not `/kaggle/working`?**  
> Kaggle enforces a **20 GB quota** on `/kaggle/working/` (the output directory). Our model alone weighs 21.6 GB — it doesn't fit.  
> `/tmp` shares the full disk (≈73 GB) with no per-folder quota, so the download fits comfortably.  
> We don't need the files to survive the session — once the Kaggle Dataset is created, the local copy is no longer needed.

In [ ]:
import os
import json
from pathlib import Path

# 高速传输：新版 huggingface_hub 用 Xet 后端，HF_HUB_ENABLE_HF_TRANSFER 已废弃（会打 FutureWarning）
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)   # 清掉容器里可能残留的旧变量

# ── Configuration ──────────────────────────────────────────────────────────────
REPO_ID      = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF"   # 去审查版 (abliterated)
QUANT        = "UD-Q5_K_XL"                       # 量化档：19.25 GB，2x T4 上最划算
DOWNLOAD_DIR = Path("/tmp/model")                  # 落盘位置
# ⚠️  用 /tmp 不用 /kaggle/working：Kaggle 对 /kaggle/working 有 20 GB 硬配额，
# 而模型本身 19.25 GB + mmproj 0.87 GB 就顶满了。/tmp 共享整块 ~73 GB 盘，没这个限制。

# Dataset name that will appear on your Kaggle account
# 只能小写字母 + 数字 + 连字符
DATASET_SLUG = "qwen38-27b-abliterated-gguf"
# ───────────────────────────────────────────────────────────────────────────────

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Download directory: {DOWNLOAD_DIR}")
print(f"🤗 Source repo      : {REPO_ID}")
print(f"🗜️  Quantization     : {QUANT}")
print(f"📦 Dataset slug     : {DATASET_SLUG}")

## 📥 Cell 3 — Download model files from HuggingFace

We use `hf_hub_download` with glob filters to grab only the two files we need:
- `*UD-Q4_K_XL*` → the main model
- `*mmproj-F16*` → the vision projector

This avoids downloading the entire repo (which includes other quants we don't need).

> ⏳ Estimated download time: **15–30 minutes** depending on Kaggle's connection speed.

In [ ]:
from huggingface_hub import snapshot_download

print("🚀 Starting download from HuggingFace...")
print("   This will take 15–30 minutes. You can monitor progress below.\n")

snapshot_download(
    repo_id   = REPO_ID,
    repo_type = "model",
    local_dir = str(DOWNLOAD_DIR),
    allow_patterns = [
        f"*{QUANT}*",   # 主模型 (~19.3 GB)
        "*mmproj*"      # 视觉投影层 (~0.87 GB)  ← 名字是 mmproj-model-bf16.gguf
    ],
)

print("\n✅ Download complete.")

## 🔍 Cell 4 — Verify downloaded files

Before creating the dataset, we verify that both files are present and have a reasonable size.

Expected sizes:
- Main model: ~22 GB
- mmproj: ~1 GB

In [ ]:
print("📋 Downloaded files:\n")

total_size = 0
found_model  = False
found_mmproj = False

for f in sorted(DOWNLOAD_DIR.rglob("*.gguf")):
    size_gb = f.stat().st_size / (1024 ** 3)
    total_size += size_gb
    tag = ""

    if QUANT in f.name:
        found_model = True
        tag = "← main model"
    if "mmproj" in f.name:
        found_mmproj = True
        tag = "← vision projector"

    print(f"  {f.name:<60} {size_gb:6.2f} GB  {tag}")

print(f"\n  Total: {total_size:.2f} GB")
print()

# Sanity checks
assert found_model,  "❌ Main model file not found! Check REPO_ID / QUANT."
assert found_mmproj, "❌ mmproj file not found! Multimodal will not work."
assert total_size > 18, f"❌ Total only {total_size:.2f} GB — expected ~20 GB (Q5_K_XL 19.25 + mmproj 0.87)."
print("✅ Both files verified — ready to create the Kaggle Dataset.")

## 📦 Cell 5 — Create the Kaggle Dataset

Kaggle requires a `dataset-metadata.json` file alongside the data files. We generate it automatically using your Kaggle username (read from the environment — Kaggle injects it automatically in notebooks).

Then we run `kaggle datasets create` to publish the dataset to your account as **private**.

> This step uploads the files to Kaggle's storage. It may take a few minutes.

> **私有 dataset 不进全局搜索。** 传完只能用 `https://www.kaggle.com/datasets/<你的用户名>/qwen38-27b-abliterated-gguf` 或者 Profile → Datasets 打开。在 Kaggle 首页搜索框里搜名字是搜不到的，这不是失败。

In [ ]:
import subprocess, json, os
from pathlib import Path

# ── Your Kaggle username ────────────────────────────────────────────────────────
# https://www.kaggle.com/settings  (under 'Username')
KAGGLE_USERNAME = ""   # ← 建议直接填死你的用户名；留空则自动去 kaggle.json / 环境变量里找
# ───────────────────────────────────────────────────────────────────────────────

KAGGLE_JSON = Path.home() / ".kaggle" / "kaggle.json"


def detect_username():
    if KAGGLE_USERNAME:
        return KAGGLE_USERNAME, "本格顶部变量"
    if os.environ.get("KAGGLE_USERNAME"):
        return os.environ["KAGGLE_USERNAME"], "环境变量 KAGGLE_USERNAME"
    if KAGGLE_JSON.exists():
        try:
            u = json.loads(KAGGLE_JSON.read_text())["username"]
            if u:
                return u, "~/.kaggle/kaggle.json"
        except Exception as e:
            print(f"⚠️  kaggle.json 读不动: {e}")
    return "", ""


kaggle_username, source = detect_username()

if not kaggle_username:
    print("❌ 找不到 Kaggle 用户名。环境自检：")
    print(f"   kaggle.json 存在    : {KAGGLE_JSON.exists()}   ({KAGGLE_JSON})")
    print(f"   KAGGLE_USERNAME 环境 : {os.environ.get('KAGGLE_USERNAME')!r}")
    kaggle_envs = {k: (v[:8] + "…" if len(v) > 8 else v) for k, v in os.environ.items() if "KAGGLE" in k.upper()}
    print(f"   所有 KAGGLE* 变量    : {kaggle_envs or '(一个都没有)'}")
    print(f"   HOME                 : {Path.home()}")
    raise ValueError(
        "KAGGLE_USERNAME 为空。\n"
        "解法：把本格最上面的 KAGGLE_USERNAME 填成你自己的用户名\n"
        "（https://www.kaggle.com/settings 页面上那个 Username）。\n"
        "Kaggle notebook 默认不注入这个环境变量，所以填死最稳。"
    )

DATASET_REF = f"{kaggle_username}/{DATASET_SLUG}"
print(f"👤 Kaggle username: {kaggle_username}   (来源: {source})")
print(f"📦 Dataset ref    : {DATASET_REF}\n")

# ── 认证检查：CLI 没 token 的话 create 一定失败，先查明 ────────────────────────
if not KAGGLE_JSON.exists():
    print("⚠️  没找到 ~/.kaggle/kaggle.json —— kaggle CLI 可能没认证。")
    print("    Kaggle notebook 里一般自带；没有就用 Add-ons → Secrets 加 KAGGLE_KEY，")
    print("    或把本机 kaggle.json 传上来。先跑下面这格看真实的 auth 状态。\n")

# ── Write dataset-metadata.json ────────────────────────────────────────────────
metadata = {
    "title"   : "Qwen3.8-27B abliterated GGUF (Q5_K_XL)",
    "id"      : DATASET_REF,
    "licenses": [{"name": "apache-2.0"}],
    "isPrivate": True
}
metadata_path = DOWNLOAD_DIR / "dataset-metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print("📝 dataset-metadata.json created.\n")


def run(args):
    r = subprocess.run(args, capture_output=True, text=True)
    print("$ " + " ".join(args))
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print("stderr: " + r.stderr.strip())
    print()
    return r


# ── 先看 CLI 版本和 auth 状态 ─────────────────────────────────────────────────
run(["kaggle", "--version"])
run(["kaggle", "config", "view"])

# ── Push dataset to Kaggle ─────────────────────────────────────────────────────
# --dir-mode skip : 原样上传，不打包。20 GB 的 gguf 再 zip 一遍既费磁盘也几乎没压缩。
print("⬆️  Uploading to Kaggle (this may take several minutes)...\n")
r = run(["kaggle", "datasets", "create", "-p", str(DOWNLOAD_DIR), "--dir-mode", "skip"])
blob = (r.stdout + r.stderr).lower()

if r.returncode != 0 and ("already exists" in blob or "409" in blob):
    print("↻ 同名 dataset 已存在，改建新版本\n")
    r = run(["kaggle", "datasets", "version", "-p", str(DOWNLOAD_DIR), "--dir-mode", "skip",
             "-m", "refresh qwen3.8-27b"])
    blob = (r.stdout + r.stderr).lower()

if r.returncode == 0:
    print("✅ 上传成功")
    print(f"🔗 https://www.kaggle.com/datasets/{DATASET_REF}")
    print("   ⚠️ 私有 dataset 不进 Kaggle 全局搜索，只能靠这个链接 / Profile → Datasets 打开")
    print("\n📋 账号下现有 dataset（核对有没有真的传上去）：")
    run(["kaggle", "datasets", "list", "--user", kaggle_username, "--page-size", "20"])
else:
    print("❌ 上传失败，原因看上面输出：")
    if "no such option" in blob and "dir-mode" in blob:
        print("   → kaggle CLI 太旧，不认 --dir-mode。")
        print("     !pip install -U -q kaggle 后重跑本格；或删掉 --dir-mode skip 两个参数重跑。")
    elif "403" in blob or "401" in blob or "unauthorized" in blob:
        print("   → 认证失败：CLI 没有有效 token。")
        print("     Add-ons → Secrets 加 KAGGLE_KEY（值是你的 API token），或上传 kaggle.json。")
    elif "quota" in blob or "limit" in blob:
        print("   → 配额满：先删账号里旧的 dataset 再传。")
    else:
        print("   → 把上面 stderr 整段贴给我。")


## ✅ 完事

模型文件已经永久存在你 Kaggle 账号里了。

### 下一步

打开 `2_run_server.ipynb`，把 dataset 挂上：

1. Kaggle 右侧 → **Add data** → 搜 `qwen38-27b-abliterated-gguf`
2. 选中你的私有 dataset
3. 文件会出现在 `/kaggle/input/datasets/<你的用户名>/qwen38-27b-abliterated-gguf/`

---

### 服务器 notebook 里要对应的两个路径

```python
MODEL_PATH  = "/kaggle/input/datasets/<user>/qwen38-27b-abliterated-gguf/Huihui-Qwen3.8-27B-abliterated-UD-Q5_K_XL.gguf"
MMPROJ_PATH = "/kaggle/input/datasets/<user>/qwen38-27b-abliterated-gguf/mmproj-model-bf16.gguf"
```

> 注意 mmproj 的文件名是 `mmproj-model-bf16.gguf`，**不是**原来的 `mmproj-F16.gguf`。